# ZEV mandate scenarios

Start from the latest registrations (2024 to now, with this year estimated), then project to the end of the scheme under different assumptions.

**To change a scenario, edit `SCENARIOS` below.** Options (see `zevm/scenarios.py`):

| option | meaning |
|---|---|
| `params` | a params file: `"vets"` (the law, to 2030), `"scenarios/indicative_to_2035"`, or your own file that `extends:` one of these |
| `zev_share` | `"target"` (default), a constant such as `0.45`, or a year table `{2027: .32, 2028: .36}` |
| `zev_share_vs_target` | added to the target, e.g. `-0.03` for 3 points short |
| `market_growth` / `market_size` | annual growth, or `{year: registrations}` |
| `nonzev_co2_change` | annual change in non-ZEV g/km, e.g. `-0.02` |
| `strategy` | `Strategy(trading="none", borrowing=False, ...)` switches off flexibilities |
| `group_zev` | how the market ZEV share splits by manufacturer: `"logit"` (default), `"scale"`, `"uniform"` |

In [ ]:
import pandas as pd
from IPython.display import display

from zevm import compliance, plots, rules, scenarios
from zevm.scenarios import Scenario, Strategy

plots.use_style()
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)

In [ ]:
VEHICLE = "car"
h = scenarios.prepare(VEHICLE)
h.notes

In [ ]:
SCENARIOS = [
    Scenario("A. On target"),
    Scenario("B. 3pp below target", zev_share_vs_target=-0.03),
    Scenario("C. 3pp below, no trading", zev_share_vs_target=-0.03, strategy=Strategy(trading="none")),
    Scenario("D. Slow: +4pp a year", zev_share={2027: .315, 2028: .355, 2029: .395, 2030: .435}),
    Scenario("E. Indicative to 2035", params="scenarios/indicative_to_2035"),
]
results = scenarios.run_many(h, SCENARIOS)
summary = scenarios.market_summary(results)
summary[["scenario", "year", "basis", "zev_share", "zev_target", "required_after_co2", "co2_credits", "traded",
         "borrowed", "bank_end", "debt_end", "shortfall", "payment_gbp_m", "paying"]]

In [ ]:
plots.scenario_lines(summary, "zev_share", "Market ZEV share by scenario", pct=True, show_target=True);

In [ ]:
plots.scenario_lines(summary, "payment_gbp_m", "Payments for unmet obligations (£m a year)", money=True, ylabel="£m");

In [ ]:
plots.bank_and_debt(summary);

In [ ]:
for name in ["B. 3pp below target", "D. Slow: +4pp a year"]:
    plots.coverage_stack(summary, name)

## Who pays?

Participants with a payment in a chosen scenario and year.

In [ ]:
SCENARIO, YEAR = "D. Slow: +4pp a year", 2029
r = results[(results.scenario == SCENARIO) & (results.year == YEAR)]
r[r.payment_gbp > 0].sort_values("payment_gbp", ascending=False)[
    ["group", "total_registrations", "zev_share", "co2_credits", "bought", "borrowed", "shortfall", "debt_defaulted", "payment_gbp"]]

## Export

In [ ]:
from pathlib import Path
out = Path("../outputs"); out.mkdir(exist_ok=True)
summary.to_csv(out / f"scenarios_{VEHICLE}_{h.as_of:%Y-%m}.csv", index=False)
results.to_csv(out / f"scenarios_{VEHICLE}_participants_{h.as_of:%Y-%m}.csv", index=False)